# qwen21-nsfw-pose · 写真套图双模式运行笔记本 (Photobook Suite Guide)

本 Notebook 专为 Google Colab 与 本地 Jupyter 打造，演示如何使用 **通用写真套图引擎（`scripts/run_photobook.py`）** 实现：
- **方式 1：T2I 文本高密度生图**（速度快、光影氛围与皮肤质感极致，适合常规写意姿态）
- **方式 2：VNCCS 3D 骨骼控制生图**（显式读取 `pose_library` 3D 骨骼模型，100% 锁定关节角度，适合瑜伽、深蹲、下腰等高难度动作）

> 参考自仓库 `colab_t4_examples.ipynb` 与 `colab_t4.ipynb` 规范，支持自动识别 Colab 环境并拉取仓库。

In [ ]:
# ===== 1. 准备：自动环境识别与路径配置（Colab / 本地自适应） =====
import os
import sys
from pathlib import Path

# 检查是否运行在 Google Colab 环境
IS_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

if IS_COLAB:
    REPO_URL = "https://github.com/ldybyz/qwen21-nsfw-pose-skill.git"
    BRANCH = "feat/colab-t4-guide"  # 合并进 main 后可改回 main
    POSELAB = "/content/qwen21-nsfw-pose-skill"

    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception:
        pass

    !test -d {POSELAB} || git clone -b {BRANCH} {REPO_URL} {POSELAB}
    !cd {POSELAB} && git fetch -q origin && git checkout -q {BRANCH} && git pull -q origin {BRANCH}

    # 切换当前工作目录到仓库根目录，并将根目录加入 sys.path
    os.chdir(POSELAB)
    if POSELAB not in sys.path:
        sys.path.insert(0, POSELAB)
    print(f"✅ [Colab] 工作目录已锁定至: {os.getcwd()}")
else:
    # 本地环境：如果当前在 notebooks/ 子目录下，自动回退到项目根目录
    cwd = Path.cwd()
    if (cwd.parent / "scripts" / "run_photobook.py").exists():
        os.chdir(cwd.parent)
    ROOT = str(Path.cwd())
    if ROOT not in sys.path:
        sys.path.insert(0, ROOT)
    print(f"✅ [本地] 工作目录已锁定至: {os.getcwd()}")

# 导入写真引擎核心方法
from scripts.run_photobook import list_available_templates, execute_photobook
print("🎉 成功导入写真引擎核心方法 execute_photobook！")

In [ ]:
# ===== 1.5 启动 ComfyUI（加 --setup-only 仅启动服务，绝不额外生成无关测试图片） =====
# 如果你是在 Colab 环境，只需跑这一行，即可完成底模配置并在后台拉起 ComfyUI：
!python scripts/colab_quickstart.py --setup-only --skip-doctor

# 验证服务端口是否已经监听
import urllib.request, time
for _ in range(15):
    try:
        with urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2):
            print("🚀 ComfyUI 后端已成功就绪，可以开始写真出图！")
            break
    except Exception:
        time.sleep(2)

In [ ]:
# ===== 2. 查看当前所有已注册的写真模板（自动标注 3D 骨骼支持状态） =====
!python scripts/run_photobook.py --list-templates

## 方式 1 示例：T2I 纯文本高密度生图

以新角色 **[CN-19] 苏青荷** 套用古典闺房拔步床模板（`chinese_canopy_bed`）为例：
- 纯文本驱动，无需 3D 浏览器，单图约 30~45 秒；
- 适合柔和自然的侧卧、回眸、慵懒伸展姿态。

In [ ]:
# ===== 3.1 方式 1（Python 方法调用）：预览苏青荷古典写真 =====
res_mode1 = execute_photobook(
    character="cn19_su_qinghe",           # 新角色：苏青荷 (古典名媛)
    template="chinese_canopy_bed",        # 古典拔步床模板
    mode="t2i",                           # 显式指定方式 1 (T2I)
    dry_run=True,                         # 预览提示词
    run=False,                            # 设为 True 则直接提交 ComfyUI 真正出图
)

In [ ]:
# ===== 3.2 方式 1（Shell 命令调用）：直接出图 =====
# 真实出图命令（输出保存在 outputs/photobooks/ 下）：
!python scripts/run_photobook.py --character cn19_su_qinghe --template chinese_canopy_bed --mode t2i --run

## 方式 2 示例：VNCCS 3D 骨骼精准控姿

以新角色 **[EA-05] 韩敏智** 套用 3D 骨骼写真模板（`pose3d_studio`）为例：
- 模板中每一幕显式绑定了 `pose_library` 里的骨骼文件（如 `Kneeling Leaning Back`、`Squatting Legs Spread` 等）；
- 3D 假人锁定四肢关节角度与相机视角，彻底消除复杂大动作的肢体畸变；
- 在 `auto` 策略下，引擎自动识别并走方式 2 渲染。

In [ ]:
# ===== 4.1 方式 2 准备：在 Colab 中安装 Playwright 无头浏览器 =====
# （仅需运行一次，用于让 VNCCS PoseStudio 在后台捕获 3D 人体骨骼画面）
!pip install -q playwright
!playwright install chromium
!playwright install-deps chromium

In [ ]:
# ===== 4.2 方式 2 执行：韩敏智 3D 精准骨骼写真出图 =====
!python scripts/run_photobook.py --character ea05_han_minji --template pose3d_studio --run

## 深入解答：方式 2 是不是不需要用提示词 txt 了？

**答案：不是！方式 2 依然必须使用提示词 txt，但提示词的分工彻底改变了！**

### 1. 为什么还要提示词？
* `pose_library` 里的 3D 木偶只是一具**“无脸、无皮肤、无环境的 3D 白模假人”**。
* 3D 骨骼只负责锁定**“骨架关节弯曲几度、相机从哪拍”**；
* 而提示词 txt 则负责告诉 AI**“这是谁（韩敏智的长相、栗色马尾）”、“在什么场景（极简采光Loft/落地窗）”、“有什么光影质感（韩国水光肌、微汗反光）”**。
* 没有提示词，3D 木偶渲染出来的就只是一具灰色白模；**3D 骨骼定四肢形体，提示词 txt 赋灵魂光影！**

### 2. 方式 1 与 方式 2 提示词的区别

| 对比项 | 方式 1 (T2I 纯文本生图) | 方式 2 (VNCCS 3D 骨骼控姿) |
| :--- | :--- | :--- |
| **四肢肢体姿势** | **必须写在提示词里**（容易写崩、出现三只手或方向反转） | **完全由 3D 假人锁定**（提示词里**无需再写**繁琐关节角度） |
| **人物身份与外貌** | 写在提示词（`{SUBJECT}` 插槽） | **依然写在提示词**（定义人物容貌与妆容发型） |
| **场景与光影环境** | 写在提示词（茶室/拔步床/烛光） | **依然写在提示词**（定义所处的空间光线） |
| **提示词文件 (.txt)** | 必须有（长达 500~1000 字符） | **必须有**（作为场景注入词传递给 VNCCS） |

In [ ]:
# ===== 5. 查看导出的方式 2 提示词与参数结构 =====
import json

param_path = Path("photobooks/han_minji_pose3d/01_kneeling_leaning_back.params.json")
txt_path = Path("photobooks/han_minji_pose3d/01_kneeling_leaning_back.txt")

if param_path.exists() and txt_path.exists():
    print("📄 [参数配置 .params.json]")
    print(json.dumps(json.loads(param_path.read_text(encoding='utf-8')), indent=2, ensure_ascii=False))
    print("\n📝 [提示词文本 .txt 内容截选]")
    print(txt_path.read_text(encoding='utf-8')[:280] + "...")
else:
    print("提示：先运行上面的第 4 步导出写真配置")

In [ ]:
# ===== 6. 查看最近生成的写真出图 =====
import glob
from IPython.display import Image, display

SHOW_LAST = 3
out_candidates = sorted(glob.glob("outputs/photobooks/**/*.png", recursive=True), key=os.path.getmtime)
if not out_candidates:
    print("还没有输出图，运行带有 --run 的命令生成图片后即可在此查看")
else:
    for p in out_candidates[-SHOW_LAST:]:
        print("显示图片:", p)
        display(Image(p, width=480))